# Fixedpoint：用递归的规则求解可达性，从建立谓词到取出路径

## 这一章要认识的引擎

Z3 中有一个名为 `Fixedpoint` 的引擎，专门用来处理递归的规则。像“从 A 能走到 B，从 B 能走到 C，因此从 A 也能走到 C”这样的推理，不必自己写循环，把规则原样交给它即可。

- 关系用规则(`rule`)与事实(`fact`)来描述
- 想知道某件事是否成立，调用 `query()` 询问
- 能够到达的时候，具体路径需要另外求出来

注意，它默认使用的引擎是自底向上的 Datalog，而由于 Datalog 与关系数据库中的查询(SQL 的递归查询)属于同一思路，因此只要把事实摆上去，它就会反复套用规则，不断推出新的事实，其过程与递归查询完全一致。

In [1]:
from z3 import *

## 例 1：图的可达性

下面看这张无向图，问题是从顶点 1 能否走到顶点 6。

```{mermaid}
flowchart LR
1 --- 2
1 --- 5
5 --- 2
4 --- 5
2 --- 3
4 --- 3
4 --- 6
```

这里要用到三个谓词，它们之间的关系是层层递进的。

- `edge(a, b)` … 表示 `a` 与 `b` 之间有一条边(作为事实给出)
- `connect(a, b)` … 表示从 `a` 能直接移到 `b`(有边则两个方向都成立)
- `path(a, b)` … 表示从 `a` 能到达 `b`(这一条由递归确定下来)

In [2]:
fp = Fixedpoint()
fp.set(engine='datalog')          # 使用 Datalog 引擎

s = BitVecSort(8)                  # 顶点编号用 8 位表示

connect = Function('connect', s, s, BoolSort())
edge = Function('edge', s, s, BoolSort())
path = Function('path', s, s, BoolSort())

a, b, c = Consts('a b c', s)

fp.register_relation(path, edge, connect)
fp.declare_var(a, b, c)

# 规则：有边就能直接移动（无向，所以两个方向都成立）
fp.rule(connect(a, b), edge(a, b))
fp.rule(connect(a, b), edge(b, a))

# 规则：能直接移动，就能到达
fp.rule(path(a, b), connect(a, b))
# 规则：a→b 能移动，且 b→c 能到达，则 a→c 能到达
fp.rule(path(a, c), [connect(a, b), path(b, c)])

# 事实：图的边
edges = [(1, 2), (1, 5), (5, 2), (4, 5), (2, 3), (4, 3), (4, 6)]
for p, q in edges:
    fp.fact(edge(p, q))

### 代码说明

第1步：挑选引擎

```python
fp = Fixedpoint()
fp.set(engine='datalog')
```

`engine='datalog'` 这一句，就是把 Datalog 引擎明确指定下来，即使不写这一句，Z3 也会选用默认的 Datalog。

第2步：指定顶点的类型

```python
s = BitVecSort(8)
```

顶点编号用 8 位的位向量存放，其取值范围是 0 到 255。由于 Datalog 只接受有限的类型，因此这里必须使用 `BitVecSort` 这样的有限类型，不能拿 `IntSort` 来代替。

第3步：定义谓词(关系)

```python
connect = Function('connect', s, s, BoolSort())
```

`Function(名称, 参数的类型..., 返回值的类型)` 用于建立谓词。connect 是接收两个顶点、返回真假的二元关系，因此返回值写成 `BoolSort()`，其中参数的类型都取自前面那个位向量类型。

第4步：声明变量、注册谓词

```python
fp.register_relation(path, edge, connect)
fp.declare_var(a, b, c)
```

谓词要先让引擎认下来，这一步由 `register_relation()` 完成；规则中要用到的变量，则由 `declare_var()` 声明。

第5步：添加规则与事实

```python
fp.rule(path(a, c), [connect(a, b), path(b, c)])
```

`rule(头, 本体)` 读作“本体全部成立时，头也成立”。上面这条规则表示：从 `a` 移到 `b`，再从 `b` 到达 `c`，因此从 `a` 也能到达 `c`。由于本体中出现了 `path` 自身，递归就是这样形成的。

至于 `fp.fact(edge(1, 2))`，它把“这里有一条边”当作事实登记进去，事实没有本体，写进去就成立，这一点与规则不同。

### 查询

In [3]:
print(fp.query(path(1, 6)))     # 能否到达
print(fp.query(path(1, 7)))     # 不存在的顶点

sat
unsat


从顶点 1 能够到达 6，因此返回 `sat`；从顶点 7 无法到达，因此返回 `unsat`。这种判断成立与否的询问，正是 Fixedpoint 的本职工作。

## 把路径重建出来

能否到达这一点已经确定，可是究竟该怎么走，仍然不知道：Z3 的 Datalog 原本带有输出说明(explanations)的功能，能够讲清某条事实为什么成立(`datalog.generate_explanations`)。

可惜在较新的 Z3 中(这个环境是 5.1.0)它并不稳定：有时报出 `explanations are not supported with undefined predicates`，有时把答案变成 `unsat`，严重时还会让 Z3 整体崩溃。

因此这一章的路径由我们自己构造：可达与否交给 Fixedpoint 判断，具体走法只要对边列表做一次 BFS(广度优先搜索)就能求出，其做法与例 1 完全相同。

In [4]:
from collections import deque


def bfs_route(edges, start, goal, undirected=True):
    """从 (from, to) 的列表出发，用 BFS 构造最短路径"""
    neighbors = {}
    for p, q in edges:
        neighbors.setdefault(p, []).append(q)
        if undirected:
            neighbors.setdefault(q, []).append(p)

    prev = {start: None}
    queue = deque([start])
    while queue:
        cur = queue.popleft()
        if cur == goal:
            break
        for nxt in neighbors.get(cur, []):
            if nxt not in prev:
                prev[nxt] = cur
                queue.append(nxt)

    if goal not in prev:
        return None

    route = [goal]
    while prev[route[-1]] is not None:
        route.append(prev[route[-1]])
    return route[::-1]


print(bfs_route(edges, 1, 6))

[1, 5, 4, 6]


算出来的路径是 1 → 5 → 4 → 6，其中 `edge(4, 5)` 是无向边，因此 5 → 4 同样走得通。

## 例 2：三个水桶

手上有 8 升、5 升、3 升三个桶，起初只有 8 升的那个是满的，另外两个空着。要让两个桶各装 4 升，需要决定倒水的顺序。

这实际上就是一张图上的可达性问题：把状态当作顶点，把倒水当作边。下面先用 Python 把所有可能的状态以及状态之间的转移全部列出来。

- 不管怎么倒，水量的合计永远是 8 升
- 能够做的只有一件事，就是从一个桶往另一个桶里倒
- 倒完之后只有两种结局：目标桶满了，或者来源桶空了

In [5]:
from itertools import product

sizes = (8, 5, 3)

# 可能的状态（合计 8 升）
states = [(a, b, c)
          for a, b, c in product(*[range(size + 1) for size in sizes])
          if a + b + c == 8]

# 一次倒水能到达的状态对
transitions = []
for s1, s2 in product(states, states):
    changed = [(size, old, new) for size, old, new in zip(sizes, s1, s2) if old != new]
    if len(changed) != 2:
        continue        # 变化的桶正好 2 个（倒出来源和倒进目标）
    if any(new == size or new == 0 for size, old, new in changed):
        transitions.append((s1, s2))    # 目标桶满 or 来源桶空

print(len(states), "种状态，", len(transitions), "种倒法")

24 种状态， 106 种倒法


下面把这些转移作为 `pour` 谓词的事实注册进去，再用 `plan` 谓词推断一件事：从初始状态出发，能否到达目标状态。

- `pour(a, b, c, a2, b2, c2)` … 表示从状态 `(a, b, c)` 一步就能移到 `(a2, b2, c2)`
- `plan(a, b, c, a2, b2, c2)` … 表示从状态 `(a, b, c)` 走若干步能够移到 `(a2, b2, c2)`

In [6]:
fp = Fixedpoint()
fp.set(engine='datalog')

s = BitVecSort(5)
pour = Function('pour', s, s, s, s, s, s, BoolSort())
plan = Function('plan', s, s, s, s, s, s, BoolSort())
a, b, c, a1, b1, c1, a2, b2, c2 = Consts(['a', 'b', 'c', 'a1', 'b1', 'c1', 'a2', 'b2', 'c2'], s)

fp.register_relation(pour, plan)
fp.declare_var(a, b, c, a1, b1, c1, a2, b2, c2)

# 一步能移，则 plan 也成立
fp.rule(plan(a, b, c, a2, b2, c2), pour(a, b, c, a2, b2, c2))
# 移一步之后还有 plan，则 plan 成立（递归）
fp.rule(plan(a, b, c, a2, b2, c2), [pour(a, b, c, a1, b1, c1), plan(a1, b1, c1, a2, b2, c2)])

for s1, s2 in transitions:
    fp.fact(pour(*s1, *s2))

start, goal = (8, 0, 0), (4, 4, 0)

%time print(fp.query(plan(*start, *goal)))

sat
CPU times: total: 0 ns
Wall time: 1.78 ms


返回 `sat`，说明从 8,0,0 到 4,4,0 之间存在可以走的路径。

路径的求法与例 1 相同，只要拿 `transitions`(那张边列表)做一次 BFS 就够了，例 1 中写好的 `bfs_route()` 可以直接复用，只是这里是有向图，参数要写成 `undirected=False`。

In [7]:
route = bfs_route(transitions, start, goal, undirected=False)

for s1, s2 in zip(route, route[1:]):
    print(s1, "->", s2)

(8, 0, 0) -> (3, 5, 0)
(3, 5, 0) -> (3, 2, 3)
(3, 2, 3) -> (6, 2, 0)
(6, 2, 0) -> (6, 0, 2)
(6, 0, 2) -> (1, 5, 2)
(1, 5, 2) -> (1, 4, 3)
(1, 4, 3) -> (4, 4, 0)


经过七次倒水，状态 4,4,0 就到达了，这个结果与人工推算的顺序一致。人类解这类问题需要反复试错，但只要状态与转移能够写出来，剩下的工作完全是机械的。

此外还可以把注册进去的规则与事实打印出来核对，这一点在调试时相当有用。

In [8]:
for rule in fp.get_rules()[:5]:
    print(rule)
print("...")

ForAll([A, B, C, D, E, F],
       Implies(pour(A, B, C, D, E, F),
               plan(A, B, C, D, E, F)))
ForAll([A, B, C, D, E, F, G, H, I],
       Implies(And(plan(D, E, F, G, H, I),
                   pour(A, B, C, D, E, F)),
               plan(A, B, C, G, H, I)))
pour(0, 5, 3, 3, 5, 0)
pour(0, 5, 3, 5, 0, 3)
pour(1, 4, 3, 0, 5, 3)
...


## 例 3：让 `pour` 变成规则

例 2 中所有转移都用 Python 算好，再把 `pour` 当作事实加入。这一次反过来，`pour` 改用规则书写：可能的状态 `state(a, b, c)` 作为事实给出，倒水的条件写成规则。

条件是两个桶的水量发生了变化，而且其中一个要么满了，要么空了。

In [9]:
fp = Fixedpoint()
fp.set(engine='datalog')

s = BitVecSort(5)
state = Function('state', s, s, s, BoolSort())
pour = Function('pour', s, s, s, s, s, s, BoolSort())
plan = Function('plan', s, s, s, s, s, s, BoolSort())
a, b, c, a1, b1, c1, a2, b2, c2 = Consts(['a', 'b', 'c', 'a1', 'b1', 'c1', 'a2', 'b2', 'c2'], s)

fp.register_relation(state, pour, plan)
fp.declare_var(a, b, c, a1, b1, c1, a2, b2, c2)

fp.rule(plan(a, b, c, a2, b2, c2), pour(a, b, c, a2, b2, c2))
fp.rule(plan(a, b, c, a2, b2, c2), [pour(a, b, c, a1, b1, c1), plan(a1, b1, c1, a2, b2, c2)])

# 倒水规则：变化的桶有 2 个，其中一个变满或变空
fp.rule(
    pour(a, b, c, a1, b1, c1),
    [
        state(a, b, c),
        state(a1, b1, c1),
        Or(
            And(a == a1, b != b1, Or(b1 == 0, b1 == 5, c1 == 0, c1 == 3)),
            And(b == b1, a != a1, Or(a1 == 0, a1 == 8, c1 == 0, c1 == 3)),
            And(c == c1, a != a1, Or(a1 == 0, a1 == 8, b1 == 0, b1 == 5)),
        ),
    ],
)

# 把可能的状态（合计 8 升）注册为事实
for p in states:
    fp.fact(state(*p))

%time print(fp.query(plan(*start, *goal)))

sat
CPU times: total: 62.5 ms
Wall time: 47.1 ms


与例 2 相同，结果仍然是 `sat`，只是求解所花的时间不同。`pour` 改用规则书写时，只要状态仍然作为事实给出，结论就完全一样，例如两次求解得到的到达步骤一致，差别只在耗时：这个环境中约为 0.04 秒，而例 2 是 1 毫秒上下。

阅读 `pour` 那条规则时，要注意本体中的 `state(a, b, c)` 与 `state(a1, b1, c1)` 两者缺一不可，而且倒水之后得到的状态，同样必须是可能的状态。

## 例 4：连 `state` 也写成规则

最后一步，把 `state` 也交给规则，不再由 Python 逐个枚举。每个桶各自能装多少，由 `bucket_a`、`bucket_b`、`bucket_c` 的事实给出，`state` 则由它们的组合(合计 8 升)推导出来。

In [10]:
fp = Fixedpoint()
fp.set(engine='datalog')

s = BitVecSort(5)
bucket_a = Function('bucket_a', s, BoolSort())
bucket_b = Function('bucket_b', s, BoolSort())
bucket_c = Function('bucket_c', s, BoolSort())
state = Function('state', s, s, s, BoolSort())
pour = Function('pour', s, s, s, s, s, s, BoolSort())
plan = Function('plan', s, s, s, s, s, s, BoolSort())
a, b, c, a1, b1, c1, a2, b2, c2 = Consts(['a', 'b', 'c', 'a1', 'b1', 'c1', 'a2', 'b2', 'c2'], s)

fp.register_relation(state, pour, plan, bucket_a, bucket_b, bucket_c)
fp.declare_var(a, b, c, a1, b1, c1, a2, b2, c2)

fp.rule(plan(a, b, c, a2, b2, c2), pour(a, b, c, a2, b2, c2))
fp.rule(plan(a, b, c, a2, b2, c2), [pour(a, b, c, a1, b1, c1), plan(a1, b1, c1, a2, b2, c2)])

fp.rule(
    pour(a, b, c, a1, b1, c1),
    [
        state(a, b, c),
        state(a1, b1, c1),
        Or(
            And(a == a1, b != b1, Or(b1 == 0, b1 == 5, c1 == 0, c1 == 3)),
            And(b == b1, a != a1, Or(a1 == 0, a1 == 8, c1 == 0, c1 == 3)),
            And(c == c1, a != a1, Or(a1 == 0, a1 == 8, b1 == 0, b1 == 5)),
        ),
    ],
)

# state 是“各桶水量的组合”，且合计为 8 升
fp.rule(state(a, b, c), [bucket_a(a), bucket_b(b), bucket_c(c), a + b + c == 8])

for i in range(sizes[0] + 1):
    fp.fact(bucket_a(i), str(i))
for i in range(sizes[1] + 1):
    fp.fact(bucket_b(i), str(i))
for i in range(sizes[2] + 1):
    fp.fact(bucket_c(i), str(i))

%time print(fp.query(plan(*start, *goal)))

sat
CPU times: total: 844 ms
Wall time: 832 ms


结果仍然是 `sat`，但慢得多：这个环境中用了 0.6 秒，而例 3 只要 0.04 秒，两者相差 15 倍左右。

原因不难想到：`state` 成为规则之后，每应用一次 `pour` 的规则，`state` 的组合都要重新考虑一遍，搜索量因此上升。能够写成规则与跑得快是两件事，这个例子正好说明：已经掌握的关系，作为事实给出更为划算，其代价则是规则的数目减少。

## 小结

| 用到的东西 | 干什么用 |
|---|---|
| `Fixedpoint()` | 创建引擎 |
| `set(engine='datalog')` | 选择 Datalog 引擎 |
| `Function(名称, 类型..., BoolSort())` | 创建谓词 |
| `register_relation(...)` | 把谓词注册到引擎 |
| `declare_var(...)` | 声明规则里用的变量 |
| `rule(头, 本体)` | 添加“本体成立则头成立”的规则 |
| `fact(...)` | 添加事实 |
| `query(...)` | 询问是否成立(`sat` / `unsat`) |
| `get_rules()` | 确认注册的规则和事实 |

- 可达性用两条规则就能写出来：“一步可以移动”加上“递归”
- Fixedpoint 只回答能否到达，具体路径需要自己用边列表做 BFS 求得
- 依靠 `generate_explanations` 取得路径的做法，在较新的 Z3 中并不稳定，不宜依赖
- 状态与转移都已知时，作为事实给出更快；全部写成规则虽然灵活，速度却会下降